## COVID Drivers: Modeling

This notebook models 
* POST_COVID
* CELL_PHONE
* IMPAIRED_DRIVER
* MATURE_DRIVER
* YOUNG_DRIVER
* FATIGUE_ASLEEP
* HIT_RUN
* UNLICENSED
* URBAN_RURALx</BR></BR>
 ~ NHTSA_AGG_DRIVING

### Table of Contents
* [Read the Data](#read)</BR>
* [Preprocessing](#prep)</BR>
* [Random Forest Classifier](#rfc)</BR>
* [Random Forest Classifier with GridSearchCV](#rf-gs)
* [Logistic Regression with Cross Validation](#lgr-cv)</BR>
* [XGBoost](#xgb)</BR>
* [XGBoost with GridSearchCV](#xgb-gs)</BR>
* [Review Models](#review)


Import packages

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime

import xgboost as xgb
from functools import reduce
#import prince

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, classification_report

In [ ]:
# Import project specific utilities
from utils.functions import *

In [ ]:
path_in = 'data/ready/ready_data.csv'

### <a id='read'>Read the data</a>

In [ ]:
df_init = pd.read_csv(path_in, low_memory=False)

In [ ]:
df_init['NHTSA_AGG_DRIVING'].sum()/df_init.shape[0]

np.float64(0.0531143466082283)

In [ ]:
df_init.loc[df_init['CRASH_DATE'] >= pd.to_datetime('2015-03-01', format='%Y-%m-%d')]['NHTSA_AGG_DRIVING'].sum()/df_init.loc[df_init['CRASH_DATE'] >= pd.to_datetime('2015-03-01', format='%Y-%m-%d')].shape[0]

np.float64(0.05537621923885981)

In [ ]:
df_init['CRASH_DATE'] = pd.to_datetime(df_init['CRASH_DATE'])

In [ ]:
df = df_init.set_index('CRASH_DATE').drop(columns=['CRN']).copy()

In [ ]:
model_metrics = []

In [ ]:
df.columns.tolist()

In [ ]:
df['URBAN_RURALx'].unique()

### <a id='prep'>Preprocessing</a>

In [ ]:
X = df.loc[:,['POST_COVID',
 'CELL_PHONE',
 'IMPAIRED_DRIVER',
 'MATURE_DRIVER',
 'YOUNG_DRIVER',
 'FATIGUE_ASLEEP',
 'HIT_RUN',
 'UNLICENSED',
 'URBAN_RURALx']].copy()

In [ ]:
y = df['NHTSA_AGG_DRIVING']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
categorical_cols = ['URBAN_RURALx']

In [ ]:
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_cols)
    ])

### <a id='rfc'>Random Forest Classifier</a>

In [ ]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('random_forest', RandomForestClassifier(random_state=42, class_weight='balanced_subsample'))
])

In [ ]:
rf_pipeline.fit(X_train, y_train)

In [ ]:
y_pred = rf_pipeline.predict(X_test)


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')

In [ ]:
conf_matrix = confusion_matrix(y_test, y_pred)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='g', cmap='Blues', cbar=False)

plt.title('Confusion Matrix Heatmap')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

In [ ]:
# Predicted probabilities for the class 1
y_pred_proba = rf_pipeline.predict_proba(X_test)[:, 1]

In [ ]:
# Precision: tp / (tp + fp)
precision = precision_score(y_test, y_pred)

In [ ]:
# Recall: tp / (tp + fn)
recall = recall_score(y_test, y_pred)

In [ ]:
# F1-score: harmonic mean of precision and recall
f1 = f1_score(y_test, y_pred)

In [ ]:
# ROC AUC: Area Under the Receiver Operating Characteristic Curve
roc_auc = roc_auc_score(y_test, y_pred_proba)

In [ ]:
measure_titles = ['Accuracy','Precision','Recall','F1 Score','ROC AUC']
measure_value = [accuracy,precision,recall,f1,roc_auc]
model = 'RandomForestClassifier'

In [ ]:
aggdrv_rf = pd.DataFrame({'Measure':measure_titles,
                            model:measure_value})

In [ ]:
aggdrv_rf['RandomForestClassifier'] = [round(x, 4) for x in aggdrv_rf['RandomForestClassifier']]

In [ ]:
aggdrv_rf

In [ ]:
model_metrics.append(aggdrv_rf)

In [ ]:
# Comprehensive classification_report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

### <a id='rf-gs'>Random Forest Classifier with GridSearchCV</a>

In [ ]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 5],
    'min_samples_split': [2, 5, 8]
}

In [ ]:
grid_search = make_pipeline(preprocessor,
                    GridSearchCV(RandomForestClassifier(random_state=42, class_weight='balanced_subsample'),
                                 param_grid=param_grid,
                                 cv=5,
                                 scoring='f1',
                                 return_train_score=True,
                                 refit=True))

In [ ]:
grid_search.fit(X_train, y_train)

In [ ]:
y_pred = grid_search.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')

In [ ]:
# Predicted probabilities for the class 1
y_pred_proba = grid_search.predict_proba(X_test)[:, 1]

In [ ]:
# Precision: tp / (tp + fp)
precision = precision_score(y_test, y_pred)

In [ ]:
# Recall: tp / (tp + fn)
recall = recall_score(y_test, y_pred)

In [ ]:
# F1-score: harmonic mean of precision and recall
f1 = f1_score(y_test, y_pred)

In [ ]:
# ROC AUC: Area Under the Receiver Operating Characteristic Curve
roc_auc = roc_auc_score(y_test, y_pred_proba)

In [ ]:
measure_titles = ['Accuracy','Precision','Recall','F1 Score','ROC AUC']
measure_value = [accuracy,precision,recall,f1,roc_auc]
model = 'RandomForestClassifier_GridSearchCV'

In [ ]:
aggdrv_rfgs = pd.DataFrame({'Measure':measure_titles,
                            model:measure_value})

In [ ]:
aggdrv_rfgs['RandomForestClassifier_GridSearchCV'] = [round(x, 4) for x in aggdrv_rfgs['RandomForestClassifier_GridSearchCV']]

In [ ]:
aggdrv_rfgs

In [ ]:
model_metrics.append(aggdrv_rfgs)

In [ ]:
# Comprehensive classification_report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

### <a id='lgr-cv'>Logistic Regression with Cross Validation</a>

In [ ]:
lgr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('logistic_regression', LogisticRegressionCV(class_weight='balanced', random_state=42, cv=5, solver='saga', l1_ratios=[0.1, 0.25, 0.5, 0.75, 0.9], use_legacy_attributes=False, max_iter=5000))
])

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
lgr_pipeline = Pipeline(steps=[
    ('logistic_regression', LogisticRegressionCV(class_weight='balanced', random_state=42, cv=skf, solver='saga', l1_ratios=[0.9], use_legacy_attributes=False, max_iter=5000))
])

In [ ]:
lgr_pipeline.fit(X_train, y_train)

In [ ]:
y_pred = lgr_pipeline.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')

In [ ]:
conf_matrix = confusion_matrix(y_test, y_pred)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='g', cmap='Blues', cbar=False)

plt.title('Confusion Matrix Heatmap')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

In [ ]:
# Predicted probabilities for the class 1
y_pred_proba = lgr_pipeline.predict_proba(X_test)[:, 1]

In [ ]:
# Precision: tp / (tp + fp)
precision = precision_score(y_test, y_pred)

In [ ]:
# Recall: tp / (tp + fn)
recall = recall_score(y_test, y_pred)

In [ ]:
# F1-score: harmonic mean of precision and recall
f1 = f1_score(y_test, y_pred)

In [ ]:
# ROC AUC: Area Under the Receiver Operating Characteristic Curve
roc_auc = roc_auc_score(y_test, y_pred_proba)

In [ ]:
measure_titles = ['Accuracy','Precision','Recall','F1 Score','ROC AUC']
measure_value = [accuracy,precision,recall,f1,roc_auc]
model = 'LogisticRegressionCV'

In [ ]:
aggdrv_lgr = pd.DataFrame({'Measure':measure_titles,
                            model:measure_value})

In [ ]:
aggdrv_lgr['LogisticRegressionCV'] = [round(x, 4) for x in aggdrv_lgr['LogisticRegressionCV']]

In [ ]:
aggdrv_lgr

In [ ]:
model_metrics.append(aggdrv_lgr)

In [ ]:
# Comprehensive classification_report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

### <a id='xgb'>XGBoost</a>

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
scale_pos_weight

In [ ]:
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('xgboost', xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        objective='binary:logistic',
        scale_pos_weight=scale_pos_weight
     ))
])

In [ ]:
xgb_pipeline.fit(X_train, y_train)

In [ ]:
y_pred = xgb_pipeline.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')

In [ ]:
# Predicted probabilities for the class 1
y_pred_proba = xgb_pipeline.predict_proba(X_test)[:, 1]

In [ ]:
# Precision: tp / (tp + fp)
precision = precision_score(y_test, y_pred)

In [ ]:
# Recall: tp / (tp + fn)
recall = recall_score(y_test, y_pred)

In [ ]:
# F1-score: harmonic mean of precision and recall
f1 = f1_score(y_test, y_pred)

In [ ]:
# ROC AUC: Area Under the Receiver Operating Characteristic Curve
roc_auc = roc_auc_score(y_test, y_pred_proba)

In [ ]:
measure_titles = ['Accuracy','Precision','Recall','F1 Score','ROC AUC']
measure_value = [accuracy,precision,recall,f1,roc_auc]
model = 'XGBoost'

In [ ]:
aggdrv_xgb = pd.DataFrame({'Measure':measure_titles,
                            model:measure_value})

In [ ]:
aggdrv_xgb['XGBoost'] = [round(x, 4) for x in aggdrv_xgb['XGBoost']]

In [ ]:
aggdrv_xgb

In [ ]:
model_metrics.append(aggdrv_xgb)

### <a id='xgb-gs'>XGBoost with GridSearchCV</a>

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
scale_pos_weight = [scale_pos_weight]

In [ ]:
# Number of trees in random forest
n_estimators = [100, 1000]
# Maximum number of levels in tree
max_depth = [None, 5, 10]
# Step size at each boosting iteration
learning_rate = [0.01, 0.2]
# Minimum sum of instance weight (hessian) required in a child node
min_child_weight = [0, 1]
# Minimum loss reduction required to make a further partition on a leaf node of the tree - gamma
min_split_loss = [0, 0.1]
# Method of selecting samples for training each tree
subsample = [0.5, 1]
# L2 regularization term on weights
reg_lambda = [1, 2]
# L1 regularization term on weights 
reg_alpha = [0, 1]


# Create the random grid
param_grid = {'clf__n_estimators': n_estimators,
               'clf__max_depth': max_depth,
               'clf__learning_rate': learning_rate,
               'clf__min_child_weight': min_child_weight,
               'clf__min_split_loss': min_split_loss,
               'clf__subsample': subsample,
               'clf__reg_lambda': reg_lambda,
               'clf__reg_alpha': reg_alpha,
               'clf__scale_pos_weight': scale_pos_weight}

In [ ]:
clf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', xgb.XGBClassifier(random_state=42))
])

In [ ]:
grid_search = GridSearchCV(clf_pipeline,
                            param_grid=param_grid,
                            cv=10,
                            scoring='f1',
                            return_train_score=True,
                            refit=True,
                            verbose=1)

In [ ]:
"""
param_grid = {'clf__max_depth': max_depth,
               'clf__learning_rate': learning_rate,
               'clf__min_child_weight': min_child_weight,
               'clf__min_split_loss': min_split_loss,
               'clf__subsample': subsample,
               'clf__reg_lambda': reg_lambda,
               'clf__reg_alpha': reg_alpha,
               'clf__scale_pos_weight': scale_pos_weight}

clf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', xgb.XGBClassifier(n_estimators=1000, early_stopping_rounds=10, random_state=42))
])

grid_search = GridSearchCV(clf_pipeline,
                            param_grid=param_grid,
                            cv=10,
                            scoring='f1',
                            return_train_score=True,
                            refit=True,
                            verbose=1)

grid_search.fit(
    X_train, y_train,
    clf__eval_set=[(X_test, y_test)]
)
"""
# fails because X_test is not one-hot encoded

In [ ]:
"""grid_search = make_pipeline(preprocessor,
                    GridSearchCV(xgb.XGBClassifier(early_stopping_rounds=10),
                                 param_grid=param_grid,
                                 cv=10,
                                 scoring='f1',
                                 return_train_score=True,
                                 refit=True,
                                 verbose=1))"""

In [ ]:
#grid_search.fit(X_train, y_train)

In [ ]:
#y_pred = grid_search.predict(X_test)

In [ ]:
#accuracy = accuracy_score(y_test, y_pred)
#print(f'Accuracy: {accuracy * 100:.2f}%')

In [ ]:
# Predicted probabilities for the class 1
#y_pred_proba = grid_search.predict_proba(X_test)[:, 1]

In [ ]:
# Precision: tp / (tp + fp)
#precision = precision_score(y_test, y_pred)

In [ ]:
# Recall: tp / (tp + fn)
#recall = recall_score(y_test, y_pred)

In [ ]:
# F1-score: harmonic mean of precision and recall
#f1 = f1_score(y_test, y_pred)

In [ ]:
# ROC AUC: Area Under the Receiver Operating Characteristic Curve
#roc_auc = roc_auc_score(y_test, y_pred_proba)

In [ ]:
"""measure_titles = ['Accuracy','Precision','Recall','F1 Score','ROC AUC']
measure_value = [accuracy,precision,recall,f1,roc_auc]
model = 'XGBoost_GridSearchCV'"""

In [ ]:
"""aggdrv_xgbgs = pd.DataFrame({'Measure':measure_titles,
                            model:measure_value})"""

In [ ]:
#aggdrv_xgbgs['XGBoost_GridSearchCV'] = [round(x, 4) for x in aggdrv_xgbgs['XGBoost_GridSearchCV']]

In [ ]:
#aggdrv_xgbgs

In [ ]:
#model_metrics.append(aggdrv_xgbgs)

### <a id='review'>Review Models</a>

In [ ]:
merged_metrics = reduce(lambda left, right: pd.merge(left, right, on='Measure', how='inner'), model_metrics)

In [ ]:
merged_metrics

In [ ]:
merged_metrics.to_csv('data/model_metrics/metrics_09_models_10x.csv', index=False)